# Filter reliable perturbations before embedding analysis

This notebook applies the initial QC criteria to the published differential-expression summary, then restricts the gene embedding to targets that pass. Targets are joined by Ensembl gene ID (`target_contrast` in the DE table and `ensembl_gene_id` in the embedding), never by gene symbol. The source CSV is used when it is a valid table; if the copy in `data_dir` is an HTML download page, the equivalent `.obs` table from `GWCD4i.DE_stats.h5ad` is used instead.

Core eligibility requires all three conditions (`Rest`, `Stim8hr`, `Stim48hr`), at least 75 targeted cells in every condition, significant on-target knockdown in at least one condition, no distal off-target flag, and representation in the selected gene embedding. Targets with at least two guides are preferred. Positive cross-guide and cross-donor correlations are also used to rank candidates, while the number of significant conditions is shown so stronger, replicated knockdown can be prioritized.

A later cell queries MyGene.info for human Gene Ontology biological-process (GO:BP) terms using the Ensembl IDs of the QC-passing targets. It writes both a one-row-per-target annotation summary and a long table with one row per target-term pair.


In [33]:
from pathlib import Path
import anndata as ad
import numpy as np
import pandas as pd

# Data directory supplied for this analysis.
data_dir = Path('/work/magroup/skrieger/Syndri/perturb')
stats_csv = data_dir / 'DE_stats.suppl_table.csv'
stats_h5ad = data_dir / 'GWCD4i.DE_stats.h5ad'
embedding_path = data_dir / 'scGenePT_GO_BP_(D=1536).h5ad'

conditions = {'Rest', 'Stim8hr', 'Stim48hr'}
min_cells_per_condition = 75
preferred_min_guides = 2
required_stats = {
    'target_contrast', 'target_contrast_gene_name', 'culture_condition', 'n_cells_target', 'n_guides',
    'ontarget_significant', 'distal_offtarget_flag', 'n_downstream',
    'guide_correlation_all', 'donor_correlation_all_mean',
}

# Prefer the supplementary CSV, but reject HTML/error pages and use the matching
# AnnData observation metadata when the local CSV is not actually tabular.
use_csv = False
if stats_csv.exists():
    try:
        stats = pd.read_csv(stats_csv)
        use_csv = required_stats.issubset(stats.columns)
    except (pd.errors.ParserError, UnicodeDecodeError):
        use_csv = False

if use_csv:
    print(f'Loaded QC table from {stats_csv}')
else:
    if not stats_h5ad.exists():
        raise FileNotFoundError(f'Neither a valid stats CSV nor {stats_h5ad} is available')
    stats_adata = ad.read_h5ad(stats_h5ad, backed='r')
    stats = stats_adata.obs.reset_index()
    # Some AnnData versions name the observation key differently; it is not
    # needed for target-condition QC.
    print(f'Loaded QC metadata from {stats_h5ad} (.obs); local CSV was not a valid table')

missing = required_stats.difference(stats.columns)
if missing:
    raise ValueError(f'The DE summary is missing required QC columns: {sorted(missing)}')

stats.head()


Loaded QC metadata from /work/magroup/skrieger/Syndri/perturb/GWCD4i.DE_stats.h5ad (.obs); local CSV was not a valid table


,index,target_contrast_gene_name,culture_condition,target_contrast,chunk,n_cells_target,n_up_genes,n_down_genes,n_total_de_genes,ontarget_effect_size,...,n_downstream,guide_correlation_signif,guide_correlation_signif_pval,guide_correlation_all,guide_correlation_all_pval,guide_n_signif_ontarget,donor_correlation_all_mean,donor_correlation_all_min,donor_correlation_hits_mean,donor_correlation_hits_min
0,ENSG00000121410_Rest,A1BG,Rest,ENSG00000121410,530,734.0,1,0,1,0.0,...,1,-1.000000,1.000000,-0.016747,7.229013e-02,0.0,NaN,NaN,NaN,NaN
1,ENSG00000121410_Stim48hr,A1BG,Stim48hr,ENSG00000121410,166,825.0,0,0,0,0.0,...,0,NaN,NaN,0.040246,1.556695e-05,0.0,NaN,NaN,NaN,NaN
2,ENSG00000121410_Stim8hr,A1BG,Stim8hr,ENSG00000121410,645,753.0,1,0,1,0.0,...,1,NaN,NaN,0.042464,5.141925e-06,0.0,NaN,NaN,NaN,NaN
3,ENSG00000175899_Rest,A2M,Rest,ENSG00000175899,127,366.0,56,66,122,0.0,...,122,-0.466767,0.147797,0.043526,2.968244e-06,0.0,0.050948,0.033937,0.814637,0.804117
4,ENSG00000175899_Stim48hr,A2M,Stim48hr,ENSG00000175899,415,236.0,32,13,45,0.0,...,45,0.351074,0.012426,0.113213,3.597323e-34,0.0,0.028927,0.012727,0.907426,0.852365


In [34]:
# Normalize source fields and compute a transparent target-level QC table.
qc = stats.copy()
qc['target_contrast'] = (
    qc['target_contrast'].astype('string').str.strip()
    .str.replace(r'\.\d+$', '', regex=True)
)
qc['target_contrast_gene_name'] = qc['target_contrast_gene_name'].astype('string').str.strip()
qc['culture_condition'] = qc['culture_condition'].astype('string').str.strip()
for col in ['n_cells_target', 'n_guides', 'n_downstream',
            'guide_correlation_all', 'donor_correlation_all_mean']:
    qc[col] = pd.to_numeric(qc[col], errors='coerce')

def as_bool(series):
    """Parse booleans from native bools or common CSV string encodings."""
    if pd.api.types.is_bool_dtype(series):
        return series.astype('boolean')
    normalized = series.astype('string').str.strip().str.lower()
    return normalized.map({'true': True, '1': True, 'yes': True,
                           'false': False, '0': False, 'no': False}).astype('boolean')

qc['ontarget_significant'] = as_bool(qc['ontarget_significant'])
qc['distal_offtarget_flag'] = as_bool(qc['distal_offtarget_flag'])
qc = qc[
    qc['target_contrast'].notna() & qc['culture_condition'].isin(conditions)
].copy()

# Per-target summary across the three condition rows. Missing off-target status
# is treated conservatively as flagged; missing correlation is not positive.
summary = qc.groupby('target_contrast').agg(
    target_gene_name=('target_contrast_gene_name', 'first'),
    conditions_present=('culture_condition', 'nunique'),
    min_cells=('n_cells_target', 'min'),
    min_guides=('n_guides', 'min'),
    significant_conditions=('ontarget_significant', lambda x: int(x.fillna(False).sum())),
    any_distal_offtarget=('distal_offtarget_flag', lambda x: bool(x.isna().any() or x.fillna(True).any())),
    mean_n_downstream=('n_downstream', 'mean'),
    positive_guide_conditions=('guide_correlation_all', lambda x: int((x > 0).sum())),
    positive_donor_conditions=('donor_correlation_all_mean', lambda x: int((x > 0).sum())),
    mean_guide_correlation=('guide_correlation_all', 'mean'),
    mean_donor_correlation=('donor_correlation_all_mean', 'mean'),
).reset_index()

summary['has_all_conditions'] = summary['conditions_present'].eq(len(conditions))
summary['cell_coverage_pass'] = summary['min_cells'].ge(min_cells_per_condition)
summary['on_target_pass'] = summary['significant_conditions'].ge(1)
summary['off_target_pass'] = ~summary['any_distal_offtarget']
summary['guide_preference_pass'] = summary['min_guides'].ge(preferred_min_guides)

# Match the Ensembl identifier in the DE table to the embedding metadata.
embedding = ad.read_h5ad(embedding_path)
embedding.obs['ensembl_gene_id'] = embedding.obs.index

embedding_id_col = 'ensembl_gene_id'
if embedding_id_col not in embedding.obs:
    raise KeyError(
        f"Expected an Ensembl ID column '{embedding_id_col}' in embedding observations: {embedding_path}"
    )
embedding.obs[embedding_id_col] = (
    embedding.obs[embedding_id_col].astype('string').str.strip()
    .str.replace(r'\.\d+$', '', regex=True)
)
embedding_ids = set(embedding.obs[embedding_id_col].dropna())
summary['in_embedding'] = summary['target_contrast'].isin(embedding_ids)

summary['passes_core_qc'] = (
    summary['has_all_conditions'] & summary['cell_coverage_pass'] &
    summary['on_target_pass'] & summary['off_target_pass'] & summary['in_embedding']
)
# Two or more guides are a stated preference. Keep this visible as its own flag
# so users can compare the broader eligible set with the preferred subset.
summary['passes_preferred_qc'] = summary['passes_core_qc'] & summary['guide_preference_pass']
summary = summary.sort_values(
    ['passes_preferred_qc', 'passes_core_qc', 'significant_conditions',
     'positive_donor_conditions', 'positive_guide_conditions', 'mean_n_downstream'],
    ascending=[False, False, False, False, False, False], na_position='last'
).reset_index(drop=True)

qc_decisions = summary
eligible_targets = qc_decisions.loc[qc_decisions['passes_core_qc'], 'target_contrast'].tolist()
preferred_targets = qc_decisions.loc[qc_decisions['passes_preferred_qc'], 'target_contrast'].tolist()

print(f"Targets represented in all three conditions and passing core QC: {len(eligible_targets)}")
print(f"Of these, targets with at least {preferred_min_guides} guides in every condition: {len(preferred_targets)}")
print('Top eligible targets (higher significance and positive reproducibility correlations rank first):')
display(qc_decisions.loc[qc_decisions['passes_core_qc']].head(20))


Targets represented in all three conditions and passing core QC: 6773
Of these, targets with at least 2 guides in every condition: 6153
Top eligible targets (higher significance and positive reproducibility correlations rank first):


,target_contrast,target_gene_name,conditions_present,min_cells,min_guides,significant_conditions,any_distal_offtarget,mean_n_downstream,positive_guide_conditions,positive_donor_conditions,mean_guide_correlation,mean_donor_correlation,has_all_conditions,cell_coverage_pass,on_target_pass,off_target_pass,guide_preference_pass,in_embedding,passes_core_qc,passes_preferred_qc
0,ENSG00000173011,TADA2B,3,1178.0,2.0,3,False,5286.000000,3,3,0.659323,0.577906,True,True,True,True,True,True,True,True
1,ENSG00000176476,SGF29,3,993.0,2.0,3,False,4120.333333,3,3,0.562633,0.455366,True,True,True,True,True,True,True,True
2,ENSG00000130165,ELOF1,3,1723.0,2.0,3,False,3922.666667,3,3,0.283441,0.330383,True,True,True,True,True,True,True,True
3,ENSG00000162227,TAF6L,3,672.0,2.0,3,False,3830.000000,3,3,0.565695,0.451129,True,True,True,True,True,True,True,True
4,ENSG00000119760,SUPT7L,3,2293.0,2.0,3,False,3634.000000,3,3,0.366340,0.296063,True,True,True,True,True,True,True,True
5,ENSG00000176208,ATAD5,3,1182.0,2.0,3,False,3620.000000,3,3,0.595266,0.502014,True,True,True,True,True,True,True,True
6,ENSG00000102710,SUPT20H,3,731.0,2.0,3,False,3443.000000,3,3,0.422193,0.376418,True,True,True,True,True,True,True,True
7,ENSG00000152382,TADA1,3,372.0,2.0,3,False,3234.666667,3,3,0.479997,0.424942,True,True,True,True,True,True,True,True
8,ENSG00000112237,CCNC,3,3051.0,2.0,3,False,3030.000000,3,3,0.185006,0.251341,True,True,True,True,True,True,True,True
9,ENSG00000170322,NFRKB,3,1321.0,2.0,3,False,2809.666667,3,3,0.399275,0.389578,True,True,True,True,True,True,True,True


In [35]:
# Keep only genes whose Ensembl IDs pass QC and occur in the embedding.
adata = embedding[embedding.obs['ensembl_gene_id'].isin(eligible_targets)].copy()
adata.obs_names = adata.obs['ensembl_gene_id'].astype(str)
adata.obs['passes_preferred_qc'] = adata.obs['ensembl_gene_id'].isin(preferred_targets).to_numpy()

print(f'Embedding rows retained: {adata.n_obs} / {embedding.n_obs}')
adata.obs.head()


Embedding rows retained: 6773 / 17017


,symbol,ncbi_gene_id,biotype,description,synonyms,ensembl_gene_id,passes_preferred_qc
ensembl_gene_id,,,,,,,
ENSG00000000419,DPM1,8813,protein_coding,dolichyl-phosphate mannosyltransferase subunit...,CDGIE|MPDS,ENSG00000000419,True
ENSG00000000460,FIRRM,55732,protein_coding,FIGNL1 interacting regulator of recombination ...,C1ORF112|FLJ10706|APOLO1|FLIP|MEICA1,ENSG00000000460,True
ENSG00000001036,FUCA2,2519,protein_coding,alpha-L-fucosidase 2,DJ20N2.5|MGC1314,ENSG00000001036,True
ENSG00000001084,GCLC,2729,protein_coding,glutamate-cysteine ligase catalytic subunit,GLCLC|GCS|GLCL,ENSG00000001084,False
ENSG00000001460,STPG1,90529,protein_coding,sperm tail PG-rich repeat containing 1,FLJ33340|MAPO2|C1ORF201,ENSG00000001460,True


In [ ]:
# Query MyGene.info with the QC-passing human Ensembl gene IDs.
# Run this notebook in the `rctd` conda environment, where `mygene` is installed.
from mygene import MyGeneInfo

mygene_client = MyGeneInfo()
qc_gene_ids = sorted(set(map(str, eligible_targets)))
mygene_hits = mygene_client.querymany(
    qc_gene_ids,
    scopes='ensemblgene',
    fields='symbol,go.BP',
    species='human',
    as_dataframe=False,
    verbose=False,
)

# Normalize MyGene's nested GO response into a stable long-form table.
go_records = []
mygene_symbols = {}
for hit in mygene_hits:
    query_id = str(hit.get('query', '')).strip()
    if not query_id or hit.get('notfound'):
        continue
    if hit.get('symbol'):
        mygene_symbols[query_id] = hit['symbol']

    go_block = hit.get('go', {})
    bp_terms = go_block.get('BP') if isinstance(go_block, dict) else None
    if bp_terms is None:
        bp_terms = hit.get('go.BP', [])
    if isinstance(bp_terms, dict):
        bp_terms = [bp_terms]
    for bp in bp_terms or []:
        if not isinstance(bp, dict) or not (bp.get('id') or bp.get('term')):
            continue
        go_records.append({
            'target_contrast': query_id,
            'mygene_symbol': hit.get('symbol'),
            'go_id': bp.get('id'),
            'go_bp_term': bp.get('term'),
            'evidence': bp.get('evidence'),
        })

go_bp_annotations = pd.DataFrame(
    go_records,
    columns=['target_contrast', 'mygene_symbol', 'go_id', 'go_bp_term', 'evidence'],
)
if not go_bp_annotations.empty:
    # MyGene can report the same target-term pair with multiple evidence codes.
    # Keep one row per GO term and retain the distinct evidence codes together.
    go_bp_annotations = go_bp_annotations.groupby(
        ['target_contrast', 'mygene_symbol', 'go_id', 'go_bp_term'],
        as_index=False,
        dropna=False,
    ).agg(evidence=('evidence', lambda values: '; '.join(sorted(set(values.dropna().astype(str))))))

# Keep a row for every QC-passing target, including genes with no returned BP terms.
qc_go_summary = pd.DataFrame({'target_contrast': qc_gene_ids})
qc_go_summary['mygene_symbol'] = qc_go_summary['target_contrast'].map(mygene_symbols)
if not go_bp_annotations.empty:
    term_summary = go_bp_annotations.groupby('target_contrast').agg(
        go_bp_term_count=('go_id', 'nunique'),
        go_bp_terms=('go_bp_term', lambda x: '; '.join(sorted(set(x.dropna())))),
    ).reset_index()
    qc_go_summary = qc_go_summary.merge(term_summary, on='target_contrast', how='left')
else:
    qc_go_summary['go_bp_term_count'] = 0
    qc_go_summary['go_bp_terms'] = ''
qc_go_summary['go_bp_term_count'] = qc_go_summary['go_bp_term_count'].fillna(0).astype(int)
qc_go_summary['go_bp_terms'] = qc_go_summary['go_bp_terms'].fillna('')
qc_go_summary['mygene_status'] = np.where(
    qc_go_summary['mygene_symbol'].notna(),
    np.where(qc_go_summary['go_bp_term_count'].gt(0), 'annotated', 'mapped; no GO:BP terms'),
    'not mapped by MyGene',
)

# Add the annotations to the target-level QC table and gene embedding metadata.
qc_decisions = qc_decisions.merge(qc_go_summary, on='target_contrast', how='left')
go_by_id = qc_go_summary.set_index('target_contrast')
adata.obs['mygene_symbol'] = adata.obs['ensembl_gene_id'].map(go_by_id['mygene_symbol'])
adata.obs['go_bp_term_count'] = adata.obs['ensembl_gene_id'].map(go_by_id['go_bp_term_count']).fillna(0).astype(int)
adata.obs['go_bp_terms'] = adata.obs['ensembl_gene_id'].map(go_by_id['go_bp_terms']).fillna('')

# Save reusable outputs next to the notebook's current working directory.
annotation_dir = Path.cwd() / 'qc_subset_go_bp'
annotation_dir.mkdir(parents=True, exist_ok=True)
summary_path = annotation_dir / 'qc_targets_go_bp_summary.csv'
terms_path = annotation_dir / 'qc_targets_go_bp_terms.csv'
qc_go_summary.to_csv(summary_path, index=False)
go_bp_annotations.to_csv(terms_path, index=False)
print(f'QC-passing targets queried: {len(qc_gene_ids)}')
print(f'Targets mapped by MyGene: {int(qc_go_summary["mygene_symbol"].notna().sum())}')
print(f'Targets with at least one GO:BP term: {int(qc_go_summary["go_bp_term_count"].gt(0).sum())}')
print(f'Saved {summary_path} and {terms_path}')
qc_go_summary.head()


In [1]:
# Visualize the QC-filtered gene embedding.
import scanpy as sc
import matplotlib.pyplot as plt

if adata.n_obs >= 3:
    n_pcs = min(50, adata.n_obs - 1, adata.n_vars - 1)
    sc.pp.pca(adata, n_comps=n_pcs)
    sc.pp.neighbors(adata, n_neighbors=min(15, adata.n_obs - 1), n_pcs=n_pcs,
                   metric='cosine')
    sc.tl.umap(adata)
    sc.pl.umap(adata, color='passes_preferred_qc', size=20,
               title='QC-filtered gene embedding')
else:
    print('At least three QC-passing genes are needed to compute a UMAP.')


NameError: name 'adata' is not defined